In [0]:
spark.sql("USE CATALOG dbw_dp750_lab")
spark.sql("USE SCHEMA medallion_project")

df_bronze = spark.table("dbw_dp750_lab.medallion_project.bronze_taxi_trips")
df_bronze.show(5)
print(f"Bronze row count: {df_bronze.count()}")

+--------------------+---------------------+-------------+-----------+----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|
+--------------------+---------------------+-------------+-----------+----------+-----------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022|
+--------------------+---------------------+-------------+-----------+----------+-----------+
only showing top 5 rows
Bronze row count: 21932


In [0]:
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
    .dropDuplicates()
)

df_silver.show(5)
print(f"Silver row count: {df_silver.count()}")

+--------------------+---------------------+-------------+-----------+----------+-----------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|trip_duration_minutes|
+--------------------+---------------------+-------------+-----------+----------+-----------+---------------------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|    9.366666666666667|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|    8.233333333333333|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018|   11.566666666666666|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|   14.183333333333334|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022|   30.616666666666667|
+--------------------+---------------------+-------------+-----------+--

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_dp750_lab.medallion_project.silver_taxi_trips")

print("silver table created successfully!")

silver table created successfully!


In [0]:
spark.sql("SELECT * FROM dbw_dp750_lab.medallion_project.silver_taxi_trips LIMIT 5").show()

+--------------------+---------------------+-------------+-----------+----------+-----------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|trip_duration_minutes|
+--------------------+---------------------+-------------+-----------+----------+-----------+---------------------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|    9.366666666666667|
| 2016-02-27 18:01:07|  2016-02-27 18:10:23|         2.83|       11.0|     10002|      11206|    9.266666666666667|
| 2016-02-28 15:16:44|  2016-02-28 15:35:32|          4.5|       16.5|     10282|      10069|                 18.8|
| 2016-02-04 00:22:49|  2016-02-04 00:38:19|          4.1|       15.0|     10019|      11104|                 15.5|
| 2016-02-20 19:28:12|  2016-02-20 20:08:33|         19.6|       54.5|     11422|      11238|                40.35|
+--------------------+---------------------+-------------+-----------+--

In [0]:
df_silver = (
    df_bronze
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
    .filter(F.col("trip_duration_minutes") <= 180)  # NEW: cap at 3 hours, reasonable max for a taxi trip
    .dropDuplicates()
)

df_silver.show(5)
print(f"Silver row count: {df_silver.count()}")

+--------------------+---------------------+-------------+-----------+----------+-----------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|trip_duration_minutes|
+--------------------+---------------------+-------------+-----------+----------+-----------+---------------------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|    9.366666666666667|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|    8.233333333333333|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018|   11.566666666666666|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|   14.183333333333334|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022|   30.616666666666667|
+--------------------+---------------------+-------------+-----------+--

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_dp750_lab.medallion_project.silver_taxi_trips")

print("silver table created successfully!")


silver table created successfully!
